# 04a — Feature Validation

**Week 4 support: independent check of the engineered features** · Ahamed M.A.U (IT24101779) · Group AI-39

`04_feature_engineering.ipynb` (Umair M.W.M) produced `data/processed/features.csv`. This notebook checks that file
**independently**, before any model is trained on it:

| # | Check | What it proves |
|---|---|---|
| 1 | **Independent rebuild** of all 8 base features with a *different algorithm* (cumulative sums, not `groupby().rolling()`), compared on every row | The features are computed as defined |
| 2 | **Imputed values** equal the medians of the training seasons only | Imputation used no test-season information |
| 3 | **SQL spot-checks** of individual matches, straight from the raw database | A human-readable trace of the definitions (guide §5.4) |
| 4 | **Truncation test**: rebuild with all future matches deleted | No feature uses future matches |
| 5 | **Perturbation test**: change one match's score | No feature uses its own match's result |
| 6 | **EDA signal re-check** on the final features | The final features still carry the signal found in EDA |

Checks 4 and 5 run Umair's own functions from `src/feature_engineering.py`. Checks 1–3 use separate code on purpose.

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.db import load_matches, query
from src.feature_engineering import build_team_long, add_historical_features, to_match_level

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BASE_FEATURES = [
    "Home_Form_Last_5", "Away_Form_Last_5",
    "Home_Avg_Goals_Last_5", "Away_Avg_Goals_Last_5",
    "Home_Avg_Conceded_Last_5", "Away_Avg_Conceded_Last_5",
    "Home_Win_Rate", "Away_Win_Rate",
]
TEST_SEASON = "2015/2016"          # D-021
WARMUP_SEASON = "2008/2009"        # D-019

features = pd.read_csv("../data/processed/features.csv", parse_dates=["date"]).set_index("id")
matches = load_matches(with_names=False)
print("features.csv:", features.shape, "| raw matches:", matches.shape)

features.csv: (22653, 16) | raw matches: (25979, 115)


## 2. Independent rebuild

Each base feature is recomputed from the raw goals with **cumulative sums**. For a team's *i*-th match, the points in
its previous 5 matches are `cumsum[i] − cumsum[i−5]` over the rows *before* it. The venue win rate is
`(wins so far at this venue − this match's win) / (matches so far at this venue − 1)`. This is a different algorithm
from the `shift(1).rolling(5)` used in `src/feature_engineering.py`, so agreement is meaningful.

In [2]:
# One row per team per match, from that team's point of view
sides = []
for side, opp in [("home", "away"), ("away", "home")]:
    d = matches[["id", "date", "season", f"{side}_team_api_id", f"{side}_team_goal", f"{opp}_team_goal"]].copy()
    d.columns = ["id", "date", "season", "team", "goals_for", "goals_against"]
    d["venue"] = side
    sides.append(d)
long = pd.concat(sides).sort_values(["team", "date", "id"]).reset_index(drop=True)
long["points"] = np.where(long["goals_for"] > long["goals_against"], 3,
                          np.where(long["goals_for"] == long["goals_against"], 1, 0))
long["win"] = (long["goals_for"] > long["goals_against"]).astype(int)


def sum_of_previous(values, k):
    """Sum of the k values strictly before each position; NaN when fewer than k exist."""
    cs = np.concatenate([[0], np.cumsum(values.to_numpy())])
    i = np.arange(len(values))
    return pd.Series(np.where(i >= k, cs[i] - cs[np.maximum(i - k, 0)], np.nan), index=values.index)


by_team = long.groupby("team", group_keys=False)
long["form"] = by_team["points"].apply(lambda s: sum_of_previous(s, 5))
long["avg_for"] = by_team["goals_for"].apply(lambda s: sum_of_previous(s, 5)) / 5
long["avg_against"] = by_team["goals_against"].apply(lambda s: sum_of_previous(s, 5)) / 5

by_venue = long.groupby(["team", "venue"], group_keys=False)
long["prev_venue_matches"] = long.groupby(["team", "venue"]).cumcount()
long["prev_venue_wins"] = by_venue["win"].apply(lambda s: s.cumsum() - s)
long["win_rate"] = long["prev_venue_wins"] / long["prev_venue_matches"].replace(0, np.nan)

home = long[long["venue"] == "home"].set_index("id")
away = long[long["venue"] == "away"].set_index("id")
rebuilt = pd.DataFrame({
    "Home_Form_Last_5": home["form"], "Away_Form_Last_5": away["form"],
    "Home_Avg_Goals_Last_5": home["avg_for"], "Away_Avg_Goals_Last_5": away["avg_for"],
    "Home_Avg_Conceded_Last_5": home["avg_against"], "Away_Avg_Conceded_Last_5": away["avg_against"],
    "Home_Win_Rate": home["win_rate"], "Away_Win_Rate": away["win_rate"],
    "season": home["season"], "home_prev_home": home["prev_venue_matches"], "away_prev_away": away["prev_venue_matches"],
})
rebuilt = rebuilt[rebuilt["season"] != WARMUP_SEASON]   # history uses 2008/09, rows start in 2009/10 (D-019)

assert set(rebuilt.index) == set(features.index), "Different match IDs"
print("Same 22,653 match IDs in both:", len(rebuilt) == len(features))

Same 22,653 match IDs in both: True


## 3. Compare with `features.csv`, and check the imputation

Rows where the rebuild is `NaN` are the **cold-start** rows (fewer than 5 earlier matches, or no earlier match at that
venue). `features.csv` has these **imputed**, so they are checked against the medians of the **training seasons
only** (2009/10–2014/15, D-019 / D-022).

In [3]:
csv = features.loc[rebuilt.index]
train_medians = rebuilt.loc[rebuilt["season"] != TEST_SEASON, BASE_FEATURES].median()

results = []
for col in BASE_FEATURES:
    computed = rebuilt[col].notna()
    results.append({
        "feature": col,
        "rows_computed": int(computed.sum()),
        "max_abs_difference": (rebuilt.loc[computed, col] - csv.loc[computed, col]).abs().max(),
        "rows_imputed": int((~computed).sum()),
        "train_median": train_medians[col],
        "imputed_equal_train_median": bool(np.allclose(csv.loc[~computed, col], train_medians[col])),
    })
validation = pd.DataFrame(results).set_index("feature")
validation

,rows_computed,max_abs_difference,rows_imputed,train_median,imputed_equal_train_median
feature,,,,,
Home_Form_Last_5,22377,0.000000e+00,276,7.000000,True
Away_Form_Last_5,22374,0.000000e+00,279,7.000000,True
Home_Avg_Goals_Last_5,22377,0.000000e+00,276,1.200000,True
Away_Avg_Goals_Last_5,22374,0.000000e+00,279,1.200000,True
Home_Avg_Conceded_Last_5,22377,0.000000e+00,276,1.400000,True
Away_Avg_Conceded_Last_5,22374,0.000000e+00,279,1.200000,True
Home_Win_Rate,22542,1.110223e-16,111,0.454545,True
Away_Win_Rate,22542,9.714451e-17,111,0.260870,True


In [4]:
all_seasons_medians = rebuilt[BASE_FEATURES].median()
print("Do the medians change if the test season is included?")
print((all_seasons_medians != train_medians).rename("differs").to_string())

print("\nForm_Diff = Home − Away form:", np.allclose(csv["Form_Diff"], csv["Home_Form_Last_5"] - csv["Away_Form_Last_5"]))
print("Abs_Form_Diff = |Form_Diff|: ", np.allclose(csv["Abs_Form_Diff"], csv["Form_Diff"].abs()))
print("Missing values in features.csv:", int(features.isna().sum().sum()))

Do the medians change if the test season is included?
Home_Form_Last_5            False
Away_Form_Last_5            False
Home_Avg_Goals_Last_5       False
Away_Avg_Goals_Last_5       False
Home_Avg_Conceded_Last_5    False
Away_Avg_Conceded_Last_5    False
Home_Win_Rate                True
Away_Win_Rate                True

Form_Diff = Home − Away form: True
Abs_Form_Diff = |Form_Diff|:  True
Missing values in features.csv: 0


**Result:**

- On every computed row, the independent rebuild **matches `features.csv` exactly** for all 8 features. The largest
  difference is about 1e-16, which is floating-point rounding on the win rates.
- Every imputed row equals the **training-season median** (276–279 rows per form/goal feature, 111 per win rate). These
  medians are the same values recorded in `docs/preprocessing_feature_log.md`.
- Including the test season **would change** the win-rate medians, so the training-only rule makes a real difference,
  and it has been followed.
- `Form_Diff` and `Abs_Form_Diff` are consistent with the form columns, and the file has no missing values.

## 4. SQL spot-checks (guide §5.4)

For three randomly chosen matches, the home team's previous 5 matches are pulled **straight from the raw database with
SQL**, and the features are recomputed by hand from that printout.

In [5]:
def previous_matches(team_id, before_date, n=5):
    return query("""
        SELECT date, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal
        FROM Match
        WHERE (home_team_api_id = :team OR away_team_api_id = :team) AND date < :before
        ORDER BY date DESC LIMIT :n
    """, params={"team": int(team_id), "before": before_date, "n": n})


def home_record_before(team_id, before_date):
    return query("""
        SELECT COUNT(*) AS played, SUM(home_team_goal > away_team_goal) AS won
        FROM Match WHERE home_team_api_id = :team AND date < :before
    """, params={"team": int(team_id), "before": before_date}).iloc[0]


spot_rows = []
# One random match from each of three different leagues, home team with >= 5 earlier home matches
eligible = features.loc[rebuilt.index[rebuilt["home_prev_home"] >= 5]]
sample_ids = (eligible.groupby("league_id").sample(1, random_state=42)
              .sample(3, random_state=42).index)
for match_id in sample_ids:
    row = features.loc[match_id]
    team, date = int(row["home_team_api_id"]), row["date"].strftime("%Y-%m-%d %H:%M:%S")
    prev = previous_matches(team, date)
    is_home = prev["home_team_api_id"] == team
    gf = np.where(is_home, prev["home_team_goal"], prev["away_team_goal"])
    ga = np.where(is_home, prev["away_team_goal"], prev["home_team_goal"])
    pts = np.where(gf > ga, 3, np.where(gf == ga, 1, 0))
    rec = home_record_before(team, date)

    print(f"Match {match_id} on {date[:10]}: home team {team}. Its previous 5 matches:")
    print(prev.assign(goals_for=gf, goals_against=ga, points=pts).to_string(index=False), "\n")
    spot_rows.append({
        "match_id": match_id,
        "form_by_hand": pts.sum(), "form_csv": row["Home_Form_Last_5"],
        "avg_goals_by_hand": gf.mean(), "avg_goals_csv": row["Home_Avg_Goals_Last_5"],
        "avg_conceded_by_hand": ga.mean(), "avg_conceded_csv": row["Home_Avg_Conceded_Last_5"],
        "home_win_rate_by_hand": rec["won"] / rec["played"], "home_win_rate_csv": row["Home_Win_Rate"],
    })

spot = pd.DataFrame(spot_rows).set_index("match_id")
spot

Match 14233 on 2011-11-20: home team 10217. Its previous 5 matches:
               date  home_team_api_id  away_team_api_id  home_team_goal  away_team_goal  goals_for  goals_against  points
2011-11-06 00:00:00             10229             10217               3               0          0              3       0
2011-10-29 00:00:00             10228             10217               4               0          0              4       0
2011-10-22 00:00:00             10217              8464               1               0          1              0       3
2011-10-16 00:00:00              9803             10217               4               1          1              4       0
2011-10-02 00:00:00             10235             10217               0               3          3              0       3 

Match 975 on 2011-09-17: home team 9993. Its previous 5 matches:
               date  home_team_api_id  away_team_api_id  home_team_goal  away_team_goal  goals_for  goals_against  points
2011-09-10 

,form_by_hand,form_csv,avg_goals_by_hand,avg_goals_csv,avg_conceded_by_hand,avg_conceded_csv,home_win_rate_by_hand,home_win_rate_csv
match_id,,,,,,,,
14233,6,6.0,1.0,1.0,2.2,2.2,0.375000,0.375000
975,6,6.0,1.6,1.6,1.4,1.4,0.408163,0.408163
23152,9,9.0,1.0,1.0,1.2,1.2,0.434211,0.434211


In [6]:
pairs = [("form_by_hand", "form_csv"), ("avg_goals_by_hand", "avg_goals_csv"),
         ("avg_conceded_by_hand", "avg_conceded_csv"), ("home_win_rate_by_hand", "home_win_rate_csv")]
assert all(np.allclose(spot[a], spot[b]) for a, b in pairs)
print("All SQL spot-checks match features.csv.")

All SQL spot-checks match features.csv.


All three match. Match 23152 (2012-08-25) also shows the form window **rolling across the summer break**: its last 5
include one August 2012 match and four from May 2012, as intended (roll across seasons, EDA-31).

## 5. Leakage tests on the feature code itself

These run **Umair's functions** (`build_team_long` → `add_historical_features` → `to_match_level`) on modified data.

### 5.1 Truncation test: delete the future

If a feature only uses the past, deleting every match after a cut-off date must leave the features of all earlier
matches **exactly unchanged**.

In [7]:
def build_features(df):
    team_matches = add_historical_features(build_team_long(df))
    return to_match_level(team_matches).set_index("id")[BASE_FEATURES]

all_matches = load_matches()
full = build_features(all_matches)

cutoff = pd.Timestamp("2013-01-01")
truncated = build_features(all_matches[all_matches["date"] < cutoff])
same = np.allclose(full.loc[truncated.index].to_numpy(dtype=float), truncated.to_numpy(dtype=float), equal_nan=True)
print(f"Matches before {cutoff.date()}: {len(truncated):,} — features identical with all future matches deleted: {same}")
assert same

Matches before 2013-01-01: 14,693 — features identical with all future matches deleted: True


### 5.2 Perturbation test: change one match's result

If a feature never uses its own match's result, then changing a match's score must:
- leave **that match's own features** unchanged, and
- change the features of **the team's next match**. This shows the test can actually detect a change.

In [8]:
target = all_matches[all_matches["season"] == "2012/2013"].iloc[100]
perturbed = all_matches.copy()
row = perturbed["id"] == target["id"]
perturbed.loc[row, ["home_team_goal", "away_team_goal"]] = [target["away_team_goal"] + 5, target["home_team_goal"]]
after = build_features(perturbed)

team = target["home_team_api_id"]
next_match = all_matches[(all_matches["date"] > target["date"]) &
                         ((all_matches["home_team_api_id"] == team) | (all_matches["away_team_api_id"] == team))
                         ].sort_values("date").iloc[0]["id"]

own_unchanged = np.allclose(full.loc[target["id"]].astype(float), after.loc[target["id"]].astype(float), equal_nan=True)
next_changed = not np.allclose(full.loc[next_match].astype(float), after.loc[next_match].astype(float), equal_nan=True)
print(f"Changed match {target['id']}: score {target['home_team_goal']}-{target['away_team_goal']} → "
      f"{target['away_team_goal'] + 5}-{target['home_team_goal']}")
print("Its own features unchanged:     ", own_unchanged)
print("Team's next match features changed:", next_changed)
assert own_unchanged and next_changed
pd.DataFrame({"before": full.loc[next_match], "after": after.loc[next_match]}).T

Changed match 1097: score 2-2 → 7-2
Its own features unchanged:      True
Team's next match features changed: True


,Home_Form_Last_5,Away_Form_Last_5,Home_Avg_Goals_Last_5,Away_Avg_Goals_Last_5,Home_Avg_Conceded_Last_5,Away_Avg_Conceded_Last_5,Home_Win_Rate,Away_Win_Rate
before,3.0,7.0,0.8,1.8,1.4,1.8,0.528571,0.309859
after,3.0,9.0,0.8,2.8,1.4,1.8,0.528571,0.309859


## 6. Does the final dataset still carry the EDA signal?

`02_eda.ipynb` §10 found that the home-win rate rises from ~30% to ~69% across form-difference bins (EDA-25). The same
check on the final, imputed features:

In [9]:
bins = [-16, -6, -3, -1, 1, 3, 6, 16]
labels = ["≤ −6", "−5 to −3", "−2 to −1", "0 to +1", "+2 to +3", "+4 to +6", "≥ +7"]
OUTCOMES = ["Home Win", "Draw", "Away Win"]
signal = pd.crosstab(pd.cut(features["Form_Diff"], bins=bins, labels=labels), features["match_outcome"],
                     normalize="index")[OUTCOMES] * 100
signal["matches"] = pd.cut(features["Form_Diff"], bins=bins, labels=labels).value_counts()
signal.round(1)

match_outcome,Home Win,Draw,Away Win,matches
Form_Diff,,,,
≤ −6,29.8,25.5,44.8,2945
−5 to −3,38.1,26.1,35.8,4273
−2 to −1,42.0,27.6,30.3,3599
0 to +1,45.8,28.0,26.1,3851
+2 to +3,51.7,25.2,23.1,3270
+4 to +6,56.8,22.7,20.5,3053
≥ +7,68.8,18.5,12.6,1662


## 7. Observation: win rates from short histories

`Home_Win_Rate` / `Away_Win_Rate` are expanding means, so a team with only one or two earlier matches at that venue
gets a rate of exactly 0 or 1. How common is that in the final data?

In [10]:
short = pd.DataFrame({
    "Home_Win_Rate": rebuilt["home_prev_home"].between(1, 4),
    "Away_Win_Rate": rebuilt["away_prev_away"].between(1, 4),
})
extreme = pd.DataFrame({
    "Home_Win_Rate": csv["Home_Win_Rate"].isin([0, 1]),
    "Away_Win_Rate": csv["Away_Win_Rate"].isin([0, 1]),
})
pd.DataFrame({
    "rows based on 1–4 earlier venue matches": short.sum(),
    "% of rows": (short.mean() * 100).round(1),
    "of which exactly 0 or 1": (short & extreme).sum(),
    "exactly 0 or 1 with 5+ earlier matches": (~short & extreme).sum(),
})

,rows based on 1–4 earlier venue matches,% of rows,of which exactly 0 or 1,exactly 0 or 1 with 5+ earlier matches
Home_Win_Rate,444,2.0,248,62
Away_Win_Rate,444,2.0,295,231


**Observation (not an error):** 444 rows per side (2.0%) have a win rate built from only 1–4 earlier venue matches.
These are mostly **promoted teams early in their first season**, and more than half of those rates are exactly 0 or 1
(248 home, 295 away). A rate of
0/2 says far less than 0/20, but the model sees the same value. (Rates of exactly 0 or 1 with 5+ earlier matches are
genuine, e.g. a team that has not won any of its away games yet.) Options, for Umair to decide: leave as is (trees cope well), or **shrink towards the league average**
(e.g. a Bayesian average with a prior of a few matches). Worth one decision log line either way.

## 8. Validation summary

| # | Check | Result |
|---|---|---|
| V1 | Independent rebuild (cumulative sums) vs `features.csv`, all 22,653 rows, 8 features | ✅ Exact match (max difference ≈ 1e-16) |
| V2 | Imputed cold-start values = training-season medians (2009/10–2014/15) | ✅ All 276–279 / 111 imputed rows match |
| V3 | `Form_Diff`, `Abs_Form_Diff` consistent; no missing values | ✅ |
| V4 | SQL spot-checks of 3 random matches (form, goals, conceded, home win rate) | ✅ All match |
| V5 | Truncation test: features unchanged when all matches from 2013-01-01 onwards are deleted | ✅ 14,693 matches identical |
| V6 | Perturbation test: a match's own features unchanged when its score changes; the next match's change | ✅ |
| V7 | EDA form signal reproduced on the final features | ✅ Same monotonic pattern |
| V8 | 2.0% of rows have a win rate from only 1–4 earlier venue matches, many exactly 0 or 1 | ⚠️ Observation, see §7 |

**Conclusion:** `data/processed/features.csv` is computed exactly as specified, uses **only information available
before kickoff**, and its imputation uses **training seasons only**. It is safe for Phase 5–7 modelling.